# Final independent replication analysis

This notebook reads the sealed analysis of the new 1,200-base replication. It does not rebuild the experiment. The source run completed 106,789 unique networks and 2,400 radius-one catalogue definitions.

The replication keeps the same distinction between raw binary output, one shared executable program per network, BDM on the original output matrix, and long-run dynamical observables.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'doppel-challenge/src').is_dir())
sys.path.insert(0, str(ROOT / 'doppel-challenge/src'))
from doppel_challenge.io import read_json
OUT = ROOT / 'doppel-challenge/results/joint_degree5_final_replication_analysis_v1'
release = read_json(OUT / 'release.json') if (OUT / 'release.json').exists() else {'release_ready': False, 'errors': ['release seal is created after notebook execution']}
audit = read_json(OUT / 'audit.json')
analysis = read_json(OUT / 'analysis.json')
assert audit['passed'] and audit['n_failures'] == 0
assert analysis['audit_sha256'] == audit['sha256']
groups = pd.read_csv(OUT / 'group_summary.csv')
comparison = pd.read_csv(OUT / 'comparison_with_previous.csv')
networks = pd.read_csv(OUT / 'network_metrics.csv')
effects = pd.read_csv(OUT / 'effect_rows.csv')
print('Release seal before notebook execution:', release['release_ready'])
print('Fresh replay:', audit['n_replayed'], 'networks; failures:', audit['n_failures'])
display(pd.DataFrame([{'quantity': key, 'value': analysis[key]} for key in ['n_networks', 'n_bases', 'n_catalogues', 'n_nonempty_catalogues', 'n_effects']]))

## Exact validation and denominators

The fresh audit recompiled each network, checked the serialized program, decoded every output row, replayed the exact dynamics, recomputed BDM, and compared the stored Wolfram digest. It also regenerated all catalogue IDs and exclusions and checked every constrained frontier against its saved summary.

The 2,400 catalogues are paired addition and removal catalogues for 1,200 base draws. Effects are averaged within each base and then across bases within each family, N and perturbation kind. Perturbation rows are not treated as independent network replicates.

In [ ]:
assert (networks.raw_bits == networks.n.map(lambda n: int(n) * 2**int(n))).all()
assert (networks.program_bits < networks.raw_bits).all()
assert len(groups) == 24
display(groups[['family', 'n', 'kind', 'n_bases', 'n_effects', 'mean_program_ratio', 'mean_delta_program_bits', 'mean_delta_bdm', 'mean_total_variation']].round(4))
print('Fresh catalogue frontiers checked:', audit['catalogues']['frontiers'])
print('BDM convention:', analysis['bdm'])

## Raw, program length and BDM

Raw is the already-binary ordered output matrix, with N times 2^N bits. Our program is one shared executable decision graph for the complete output repertoire; its logical length includes the declared decision and reference structure while input addresses, N and the fixed decoder are shared conventions. It is an algorithmic description length for this codec and variable order, not universal Kolmogorov complexity.

BDM is a separate model estimate in BDM units. It uses pybdm 0.1.0, 4-by-4 blocks and PartitionIgnore. N=10 requires 2,048 bottom/right padding bits; those padding bits belong to BDM's input convention and are not added to raw or program length.

In [ ]:
networks['program_raw_percent'] = 100 * networks.program_bits / networks.raw_bits
lengths = networks.groupby('n').agg(networks=('n', 'size'), raw_bits=('raw_bits', 'first'), program_min=('program_bits', 'min'), program_median=('program_bits', 'median'), program_max=('program_bits', 'max'), median_program_raw_percent=('program_raw_percent', 'median'), median_bdm=('bdm', 'median'), bdm_padding_bits=('bdm_padding_bits', 'first'))
display(lengths.round(3))
display(Image(filename=str(OUT / 'lengths_and_bdm.png')))

## Replication comparison

The following table contains the new estimate minus the estimate from the previous released study, separately for every family, network size and perturbation kind. These are descriptive replication comparisons; the studies are not pooled into a single estimate and no perturbation-level significance test is used.

In [ ]:
display(comparison[['family', 'n', 'kind', 'n_bases_replication', 'n_bases_previous', 'change_mean_program_ratio', 'change_mean_delta_program_bits', 'change_mean_delta_bdm', 'change_mean_total_variation']].round(4))
display(Image(filename=str(OUT / 'base_associations.png')))
print('All new programs shorter than raw:', analysis['all_main_programs_shorter_than_raw'])

## Scope

The new sample confirms exact reconstruction and supplies a larger descriptive replication under the same bounded construction. Ring and hub families keep fixed structural templates while varying gate assignments with seed. The maximum indegree of five is a declared sampling and resource restriction.

These results do not establish unrestricted large-network scaling, universal Kolmogorov complexity, causal mechanism identification from output distributions, or detector performance.